<a href="https://colab.research.google.com/github/tomhanna-uh/grave_d_data2026/blob/main/GRAVE_D_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GRAVE-D Dyadic TFT Forecasting – Clean A100/H100 Setup (2026)

Cell 1

Target: General multi-horizon forecasting of interstate conflict, democratization/backsliding, and intrastate conflict.
Data: GRAVE_D_Master_with_Leaders.csv (built from your repo)
Hardware: A100/H100 – minimal filtering, long sequences, high capacity

In [ ]:
# ---- Cell 2: Install & Imports (updated with miceforest) ----

!pip install -U "pytorch-forecasting[full]" lightning torch torchvision torchaudio \
    --extra-index-url https://download.pytorch.org/whl/cu121
!pip install pandas numpy matplotlib seaborn scikit-learn tqdm miceforest  # ← added here

# Rest of your imports...
import pandas as pd
import numpy as np
import torch
import lightning as L
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
# ... etc.
from pytorch_forecasting.metrics import (
    QuantileLoss, MAE, RMSE,
    CrossEntropy                      # ← use this for binary targets too
)
from pytorch_forecasting.metrics.base_metrics import MultiLoss
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.tuner import Tuner
import matplotlib.pyplot as plt
import seaborn as sns

torch.set_float32_matmul_precision('high')

In [ ]:
# ---- Cell 3: Mount Google Drive (only) ----

from google.colab import drive

# Force remount to clear any stale cache / sync issues
drive.flush_and_unmount(timeout_ms=120000)  # wait up to 2 min for clean unmount
print("Drive unmounted (if it was mounted)")

drive.mount('/content/drive', force_remount=True)
print("Drive mounted successfully.")
print("Waiting 45 seconds for filesystem sync (large file detection)...")
import time
time.sleep(45)  # ← deliberate delay; adjust to 60–90 if still issues
print("Delay complete. You can now run Cell 4.")

# ---- Cell 4: Verify path & Load large CSV ----

# ---- Cell 4: Verify path & Load large CSV ----

import os
import pandas as pd
import time

PATH = "/content/drive/MyDrive/Saved Data and Results/GRAVE_D_Master_with_Leaders.csv"

print(f"Checking path: {PATH}")

if os.path.exists(PATH):
    print("File found!")
    print(f"Size: {os.path.getsize(PATH) / (1024**3):.2f} GB")
else:
    print("File NOT found.")
    !ls -la "/content/drive/MyDrive/Saved Data and Results/" | grep -i grave
    raise FileNotFoundError("Adjust PATH.")

print("Waiting 15 seconds before reading...")
time.sleep(15)

print("Starting CSV read with latin-1 encoding...")
df = pd.read_csv(PATH, low_memory=False, encoding='latin-1')  # ← key change here
print("CSV loaded successfully.")





# Create dyad if missing
if 'dyad' not in df.columns:
    print("Creating 'dyad' column...")
    df['dyad'] = df['COWcode_a'].astype(str) + "_" + df['COWcode_b'].astype(str)

# Sort (required for TimeSeriesDataSet)
print("Sorting by dyad and year...")
df = df.sort_values(['dyad', 'year']).reset_index(drop=True)
df['year'] = df['year'].astype(int)

# Fix binary targets for CrossEntropy
binary_targets = ["mid_initiated", "nonstate_conflict_a"]
for col in binary_targets:
    if col in df.columns:
        df[col] = df[col].astype('Int64').fillna(0).astype(int)

# Print useful stats
print("\n=== Dataset Summary ===")
print(f"Total rows: {len(df):,}")
print(f"Unique dyads: {df['dyad'].nunique():,}")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")
print("mid_initiated distribution:\n", df['mid_initiated'].value_counts(dropna=False))
print("\nFirst 3 rows preview:")
print(df.head(3))

# Optional: save a small preview to verify
df.head(1000).to_csv("/content/preview.csv", index=False)
print("Saved 1000-row preview to /content/preview.csv (check Files sidebar)")

# ---- Cell 5: Full Clean Data Preparation & Imputation ----

import numpy as np
import miceforest as mf

print("=== Starting Cell 5: Full Data Cleaning & Imputation ===")

# 1. Structural zero-fill for sparse/count/ID/NAG/ATOP columns
zero_fill_keywords = [
    'atopid', 'Num_', 'NumS_', 'NumDS_', 'NumNAG_',
    'Supporter', 'Target.', 'nags_', 'Safe',
    'TrainCamp', 'HeadQuar', 'FinAid', 'WeaponLog',
    'Training', 'Troop', 'Transport'
]

zero_fill_cols = [col for col in df.columns if any(kw.lower() in col.lower() for kw in zero_fill_keywords)]

print(f"Applying structural zero-fill to {len(zero_fill_cols)} columns")
print("Examples:", zero_fill_cols[:10] if zero_fill_cols else "None found")

for col in zero_fill_cols:
    df[col] = df[col].fillna(0)

print("Structural zeros filled.")

# 2. Create any_nag_support flag
nag_count_cols = [col for col in df.columns if any(kw in col.lower() for kw in ['num_', 'nums_', 'numds_', 'numnag_'])]
if nag_count_cols:
    df['any_nag_support'] = (df[nag_count_cols].sum(axis=1) > 0).astype(int)
    print(f"Created 'any_nag_support' using {len(nag_count_cols)} columns.")
    print("Distribution:\n", df['any_nag_support'].value_counts(normalize=True))
else:
    print("No NAG count columns → skipping any_nag_support.")

# 3. Create revolutionary_leader_a and _b (must be before Colgan conditional fill)
if 'revolutionary_leader_a' not in df.columns or 'revolutionary_leader_b' not in df.columns:
    print("Creating revolutionary_leader_a and _b columns...")
    
    df['revolutionary_leader_a'] = 0
    df['revolutionary_leader_b'] = 0
    
    # Prefer direct Colgan variable if present
    if 'revolutionaryleader_a' in df.columns:
        df['revolutionary_leader_a'] = df['revolutionaryleader_a'].fillna(0).astype(int)
    elif 'revolutionaryleader' in df.columns:
        df['revolutionary_leader_a'] = df['revolutionaryleader'].fillna(0).astype(int)
        df['revolutionary_leader_b'] = df['revolutionaryleader'].fillna(0).astype(int)
    else:
        # Proxy: irregular/force entry + ≥3 institutional changes
        key_chg_a = [
            'chg_executivepower_a', 'chg_propertyowernship_a', 'chg_religioningovernment_a',
            'chg_womenandethnicstatus_a', 'chg_politicalideology_a'
        ]
        key_chg_b = [c.replace('_a', '_b') for c in key_chg_a]
        
        changes_a = df[key_chg_a].abs().sum(axis=1)
        changes_b = df[key_chg_b].abs().sum(axis=1)
        
        irregular_a = (df.get('irregulartransition', 0) > 0) | (df.get('usedforce', 0) > 0)
        irregular_b = (df.get('irregulartransition_b', 0) > 0) | (df.get('usedforce_b', 0) > 0)
        
        df['revolutionary_leader_a'] = ((irregular_a) & (changes_a >= 3)).astype(int)
        df['revolutionary_leader_b'] = ((irregular_b) & (changes_b >= 3)).astype(int)
    
    print("Revolutionary leader distribution (A side):")
    print(df['revolutionary_leader_a'].value_counts(normalize=True))
else:
    print("revolutionary_leader_a and _b already exist – skipping creation.")

# 4. Conditional zero-fill for Colgan variables
print("\nApplying conditional zero-fill for Colgan-related columns...")
colgan_keywords = ['chg_', 'seveninstitutionscoded', 'transyr', 'ftcur', 'fties', 'revolutionaryleader',
                   'radicalideology', 'foundingleader', 'foreigninstall', 'ambiguouscoding']

has_leader_a = 'revolutionary_leader_a' in df.columns
has_leader_b = 'revolutionary_leader_b' in df.columns

if not (has_leader_a and has_leader_b):
    print("Warning: revolutionary_leader_a and/or _b missing → skipping conditional Colgan fill.")
else:
    for col in df.columns:
        if any(kw in col.lower() for kw in colgan_keywords):
            if '_a' in col:
                mask = (df['revolutionary_leader_a'] == 1)
                df.loc[~mask, col] = 0
            elif '_b' in col:
                mask = (df['revolutionary_leader_b'] == 1)
                df.loc[~mask, col] = 0
            else:
                mask_either = (df['revolutionary_leader_a'] == 1) | (df['revolutionary_leader_b'] == 1)
                df.loc[~mask_either, col] = 0
    print("Colgan conditional zero-fill complete.")

# 5. Drop artifacts, Archigos metadata, redundant columns
drop_artifacts = ['version', 'ddyad']
df = df.drop(columns=[c for c in drop_artifacts if c in df.columns], errors='ignore')

drop_archigos = ['deathdate', 'deathdate_b', 'dbpedia.uri', 'dbpedia.uri_b']
df = df.drop(columns=[c for c in drop_archigos if c in df.columns], errors='ignore')

drop_redundant = ['tpop', 'flgdpen']
df = df.drop(columns=[c for c in drop_redundant if c in df.columns], errors='ignore')

print("Drops complete.")

# 6. Bandwidth filter
df = df[df['bandwidth'] > 0].copy()
print(f"\nAfter bandwidth > 0 filter: {len(df):,} rows, {df['dyad'].nunique():,} dyads")

# 7. Hybrid low-memory imputation
print("\n=== Starting Hybrid Imputation ===")

# 7.1 Zero-fill event/count columns
event_keywords = [
    'onsets', 'revonsets', 'nonrevonsets', 'sideaonsets', 'force_onsets',
    'force_sideaonsets', 'force_revonsets'
]
event_cols = [col for col in df.columns if any(kw in col.lower() for kw in event_keywords)]

for col in event_cols:
    df[col] = df[col].fillna(0)

print(f"Zero-filled {len(event_cols)} event/count columns.")

# 7.2 Group-wise ffill/bfill – safer method
print("Group-wise ffill/bfill...")
df = df.sort_values(['dyad', 'year'])
df = df.groupby('dyad', group_keys=False).apply(
    lambda g: g.ffill().bfill(), include_groups=False
).reset_index(drop=True)

print("Group fill complete.")

# 7.3 Targeted MICE on flethfrac + cinc
priority_cols = ['flethfrac', 'cinc']
priority_cols = [col for col in priority_cols if col in df.columns and df[col].isnull().any()]

if priority_cols:
    print(f"Running low-memory MICE on {len(priority_cols)} columns: {priority_cols}")
    df_impute = df[priority_cols].copy().reset_index(drop=True)
    
    kernel = mf.ImputationKernel(
        df_impute,
        save_all_iterations_data=False,
        mean_match_strategy='closest',
        random_state=1991
    )
    
    kernel.mice(4)
    df_imputed = kernel.complete_data(0)
    
    for col in priority_cols:
        df[col] = df_imputed[col].reindex(df.index).values
    
    print("Targeted MICE complete.")
else:
    print("No priority columns need MICE.")

# 7.4 Final median fill
print("\nFinal median fill for remaining numeric...")
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# 8. Final missing check
print("\n=== Final missing check ===")
final_missing = df.isnull().sum()[df.isnull().sum() > 0]
if final_missing.empty:
    print("No missing values remaining in numeric columns.")
else:
    print("Remaining missing:\n", final_missing)

print("\nCell 5 complete – ready to save to Drive.")

# ---- Cell 6: Save Cleaned & Imputed Dataset to Google Drive ----

from google.colab import drive
import os

# Remount Drive if needed (safe to run even if already mounted)
drive.mount('/content/drive', force_remount=False)

# Define save location – customize folder/filename as desired
SAVE_FOLDER = "/content/drive/MyDrive/Saved Data and Results/GRAVE_D_Cleaned"
SAVE_FILENAME = "GRAVE_D_Master_cleaned_imputed.parquet"
SAVE_PATH = f"{SAVE_FOLDER}/{SAVE_FILENAME}"

# Create the folder if it doesn't exist
os.makedirs(SAVE_FOLDER, exist_ok=True)

print(f"Saving cleaned dataset to:\n{SAVE_PATH}")
print(f"Current shape: {df.shape}")
print(f"Estimated file size (compressed): ~1.5–2.5 GB (Parquet snappy)")
print(f"Current RAM usage: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

# Save as Parquet – best format for large numeric panels
# Drop lingering object/string columns that cause Parquet issues (artifacts, leader text)
object_cols_to_drop = df.select_dtypes(include=['object']).columns.tolist()

# Optional: keep useful categoricals if any (e.g., regions)
keep_categoricals = ['unregiona', 'unregionb', 'dyad']  # add if needed
object_cols_to_drop = [col for col in object_cols_to_drop if col not in keep_categoricals]

if object_cols_to_drop:
    print(f"Dropping {len(object_cols_to_drop)} object columns to avoid Parquet errors:")
    print(object_cols_to_drop[:10], "...")
    df = df.drop(columns=object_cols_to_drop, errors='ignore')

print("Object columns cleaned – ready to save.")

df.to_parquet(
    SAVE_PATH,
    index=False,
    engine='pyarrow',
    compression='snappy'  # fast + good compression
)

print("\nSave complete!")
print(f"File saved successfully at: {SAVE_PATH}")
print("\nNext steps in a fresh runtime:")
print("1. Mount Drive")
print("2. Load with:")
print("   df = pd.read_parquet('{}')".format(SAVE_PATH))
print("3. Proceed directly to TimeSeriesDataSet creation")

In [ ]:
# ---- Cell 7: Load Cleaned & Imputed Dataset from Drive ----

from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive', force_remount=False)

LOAD_PATH = "/content/drive/MyDrive/Saved Data and Results/GRAVE_D_Cleaned/GRAVE_D_Master_cleaned_imputed.parquet"

print(f"Loading dataset from:\n{LOAD_PATH}")

if not os.path.exists(LOAD_PATH):
    raise FileNotFoundError(f"File not found at {LOAD_PATH}.")

df = pd.read_parquet(LOAD_PATH, engine='pyarrow')

# Critical fix: recreate 'dyad' if missing (from COW codes)
if 'dyad' not in df.columns:
    print("Warning: 'dyad' column missing — recreating from COWcode_a and COWcode_b")
    if 'COWcode_a' in df.columns and 'COWcode_b' in df.columns:
        df['dyad'] = df['COWcode_a'].astype(str) + "_" + df['COWcode_b'].astype(str)
        print("'dyad' recreated successfully.")
    else:
        raise KeyError("Cannot recreate 'dyad': missing both 'dyad' and 'COWcode_a'/'COWcode_b' columns.")

print("\nDataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Unique dyads: {df['dyad'].nunique():,}")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

print("\nMissing values summary:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values in any column.")

print("\nFirst 3 rows preview:")
df.head(3)

In [8]:
# ---- Cell 8: TimeSeriesDataSet and Train/Val Construction ----
from pytorch_forecasting import TimeSeriesDataSet

max_encoder_length = 20
max_prediction_length = 8

print(f"max_encoder_length: {max_encoder_length}")
print(f"max_prediction_length: {max_prediction_length}")

# Group id
group_ids = ["dyad"]
print("Using group_ids:", group_ids)

# Static dyad-level features (true static reals)
static_reals = []
if "capital_dist_km" in df.columns:
    static_reals.append("capital_dist_km")

print("Static reals:", static_reals)

# Time-varying known reals: year + bandwidths (change yearly)
candidate_tv_known = [
    "year",
    "bandwidth",
    "economicbandwidth",
    "politicalbandwidth",
    "securitybandwidth",
]
time_varying_known_reals = [c for c in candidate_tv_known if c in df.columns]
print("time_varying_known_reals:", time_varying_known_reals)

# Time-varying unknown reals: targets (and any other contemporaneous covariates you use)
candidate_tv_unknown = [
    "v2x_libdem_b",
    "mid_initiated",
    "nonstate_conflict_a",
    # add other outcome-like covariates here if you’ve been using them
]
time_varying_unknown_reals = [c for c in candidate_tv_unknown if c in df.columns]
print("time_varying_unknown_reals:", time_varying_unknown_reals)

dataset = TimeSeriesDataSet(
    df,
    time_idx="year",
    target=["v2x_libdem_b", "mid_initiated", "nonstate_conflict_a"],
    group_ids=group_ids,

    max_encoder_length=max_encoder_length,
    min_encoder_length=5,
    max_prediction_length=max_prediction_length,
    min_prediction_length=1,

    static_categoricals=[],
    static_reals=static_reals,

    time_varying_known_categoricals=[],
    time_varying_known_reals=time_varying_known_reals,

    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=time_varying_unknown_reals,

    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=True,
)

print(f"Base dataset samples: {len(dataset):,}")



max_encoder_length: 20
max_prediction_length: 8
Using group_ids: ['dyad']
Static reals: ['capital_dist_km']
time_varying_known_reals: ['year', 'bandwidth', 'economicbandwidth', 'politicalbandwidth', 'securitybandwidth']
time_varying_unknown_reals: ['v2x_libdem_b', 'mid_initiated', 'nonstate_conflict_a']


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:957: UserWarning: Target scales will be only added for continuous targets
  warnings.warn(


Base dataset samples: 2,310,578


In [12]:
# ------ Cell 8.5 ------ #

# ---- Cell 8.5: Train / Val / Test splits from dataset ----

# ---- Cell 8.5: Train / Val / Test datasets from full dataset ----

validation_years = [2010, 2012, 2014, 2016]
test_years = [2018, 2020]

print("Validation years:", validation_years)
print("Test years:", test_years)

# TRAIN = everything except validation & test years (for encoder/decoder windows)
train_df = df[~df["year"].isin(validation_years + test_years)].copy()

print(f"Train rows (raw df): {len(train_df):,}")

# Train dataset from filtered data
train_dataset = TimeSeriesDataSet.from_dataset(
    dataset,
    train_df,
    predict=False,
    stop_randomization=True,
    min_encoder_length=5,
)

print(f"Train samples: {len(train_dataset):,}")

# Val/Test datasets: use full dataset in predict=True mode,
# then we will filter by prediction years in Cell 13 / evaluation.
val_dataset = TimeSeriesDataSet.from_dataset(
    dataset,
    df,
    predict=True,
    stop_randomization=True,
)
test_dataset = val_dataset  # single prediction dataset; we’ll slice by year later

print(f"Prediction samples (all years): {len(val_dataset):,}")
print("Val/Test year filtering will happen at evaluation time.")



Validation years: [2010, 2012, 2014, 2016]
Test years: [2018, 2020]
Train rows (raw df): 1,474,930
Train samples: 2,107,298
Prediction samples (all years): 36,672
Val/Test year filtering will happen at evaluation time.


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1859: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 384 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__dyad': '100_626'}, {'__group_id__dyad': '101_626'}, {'__group_id__dyad': '110_626'}, {'__group_id__dyad': '115_626'}, {'__group_id__dyad': '130_626'}, {'__group_id__dyad': '135_626'}, {'__group_id__dyad': '140_626'}, {'__group_id__dyad': '145_626'}, {'__group_id__dyad': '150_626'}, {'__group_id__dyad': '155_626'}]
  warnings.warn(


# Helper: inspect df for plausible static dyad features
print("Columns with 'dist' or 'distance' in the name:")
dist_cols = sorted([c for c in df.columns if "dist" in c.lower() or "distance" in c.lower()])
print(dist_cols)

print("\nColumns with 'contig' or 'contiguity' in the name:")
contig_cols = sorted([c for c in df.columns if "contig" in c.lower() or "contiguity" in c.lower()])
print(contig_cols)


# ---- Cell 8b: Target Class Balance & Weight Calculation ----
import torch

print("=== Target Class Balance ===\n")

# Continuous target
print(f"v2x_libdem_b (continuous):")
print(f"  Mean: {df['v2x_libdem_b'].mean():.4f}")
print(f"  Std:  {df['v2x_libdem_b'].std():.4f}")
print(f"  Min:  {df['v2x_libdem_b'].min():.4f}")
print(f"  Max:  {df['v2x_libdem_b'].max():.4f}")

# Binary targets
for col in ['mid_initiated', 'nonstate_conflict_a']:
    counts = df[col].value_counts().sort_index()
    total = len(df)
    pos_rate = df[col].mean()
    neg_rate = 1 - pos_rate

    # Inverse frequency weighting
    weight_neg = 1.0 / neg_rate
    weight_pos = 1.0 / pos_rate
    # Normalize so negative class weight = 1.0
    weight_pos_normalized = weight_pos / weight_neg

    print(f"\n{col}:")
    print(f"  Class 0 (negative): {counts.get(0, 0):,}  ({neg_rate:.4%})")
    print(f"  Class 1 (positive): {counts.get(1, 0):,}  ({pos_rate:.4%})")
    print(f"  Imbalance ratio:    {neg_rate / pos_rate:.1f}:1")
    print(f"  Suggested weight:   torch.tensor([1.0, {weight_pos_normalized:.1f}])")

print("\n=== Copy these weights into Cell 9's CrossEntropy losses ===")


# ---- Cell 8b: Target Class Balance & Suggested Weights ----
import torch

print("=== Target Class Balance ===\n")

print("v2x_libdem_b (continuous):")
print(f"  Mean: {df['v2x_libdem_b'].mean():.4f}")
print(f"  Std:  {df['v2x_libdem_b'].std():.4f}")
print(f"  Min:  {df['v2x_libdem_b'].min():.4f}")
print(f"  Max:  {df['v2x_libdem_b'].max():.4f}")

for col in ["mid_initiated", "nonstate_conflict_a"]:
    counts = df[col].value_counts().sort_index()
    total = len(df)
    pos_rate = df[col].mean()
    neg_rate = 1 - pos_rate

    weight_neg = 1.0 / max(neg_rate, 1e-6)
    weight_pos = 1.0 / max(pos_rate, 1e-6)
    weight_pos_norm = weight_pos / weight_neg

    print(f"\n{col}:")
    print(f"  Class 0: {counts.get(0, 0):,} ({neg_rate:.4%})")
    print(f"  Class 1: {counts.get(1, 0):,} ({pos_rate:.4%})")
    print(f"  Imbalance ratio: {neg_rate / max(pos_rate, 1e-6):.1f}:1")
    print(f"  Suggested full weight: torch.tensor([1.0, {weight_pos_norm:.1f}])")

print("\nUse these as starting points (or their sqrt) in Cell 9.")


In [13]:
# ---- Cell 9: TemporalFusionTransformer Model (G4-optimized) ----
import torch, gc
import torch.nn.functional as F
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.metrics import MultiLoss, QuantileLoss, CrossEntropy

torch.cuda.empty_cache()
gc.collect()

# Custom weighted CrossEntropy for class imbalance
class WeightedCrossEntropy(CrossEntropy):
    def __init__(self, weight=None, reduction="mean", **kwargs):
        super().__init__(reduction=reduction, **kwargs)
        self._class_weight = weight

    def loss(self, y_pred, target):
        w = self._class_weight
        if w is not None:
            w = w.to(device=y_pred.device, dtype=y_pred.dtype)
        loss = F.cross_entropy(
            y_pred.view(-1, y_pred.size(-1)),
            target.view(-1).long(),
            weight=w,
            reduction="none",
        )
        return loss.view_as(target)

# Choose weights from Cell 8b output (example values)
mid_weight = torch.tensor([1.0, 30.0])        # adjust from Cell 8b
nonstate_weight = torch.tensor([1.0, 16.0])   # adjust from Cell 8b

print(f"mid_initiated weights:       {mid_weight.tolist()}")
print(f"nonstate_conflict_a weights: {nonstate_weight.tolist()}")

# Ensure no stale model
if "tft" in globals():
    del tft
torch.cuda.empty_cache()
gc.collect()
print(f"Pre-model VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

tft = TemporalFusionTransformer.from_dataset(
    dataset,
    learning_rate=0.001,
    hidden_size=512,
    lstm_layers=2,
    attention_head_size=4,
    dropout=0.25,
    hidden_continuous_size=32,
    output_size=[7, 2, 2],
    loss=MultiLoss([
        QuantileLoss(),
        WeightedCrossEntropy(weight=mid_weight),
        WeightedCrossEntropy(weight=nonstate_weight),
    ]),
    reduce_on_plateau_patience=4,
    log_interval=0,
    log_val_interval=0,
)

# Fix attention mask for bf16
tft.multihead_attn.mask_bias = -65504.0

n_params = sum(p.numel() for p in tft.parameters())
print(f"\nModel parameters: {n_params:,} ({n_params / 1e6:.1f} M)")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
print("Model ready — proceed to Cell 10.")


mid_initiated weights:       [1.0, 30.0]
nonstate_conflict_a weights: [1.0, 16.0]
Pre-model VRAM: 0.00 GB


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.



Model parameters: 17,962,341 (18.0 M)
GPU memory: 0.00 GB
Available: 101.97 GB
Model ready — proceed to Cell 10.


In [14]:
# ---- Cell 10: Trainer Setup + One-Time LR Finder ----
import os, torch, gc, pandas as pd
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, RichProgressBar, LearningRateFinder
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.tuner import Tuner
import pytorch_forecasting.data.encoders as enc
import pytorch_forecasting.metrics as met

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Safe globals for checkpoint restore (PyTorch 2.6+)
torch.serialization.add_safe_globals([
    enc.MultiNormalizer, enc.GroupNormalizer,
    met.QuantileLoss, met.CrossEntropy,
    met.base_metrics.MultiLoss,
    WeightedCrossEntropy,
    pd.core.internals.managers.BlockManager,
    pd._libs.internals._unpickle_block,
])

# Dataloaders
BATCH_SIZE = 512
NUM_WORKERS = 2

train_dataloader = train_dataset.to_dataloader(
    train=True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_dataloader = val_dataset.to_dataloader(
    train=False, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

# Callbacks
checkpoint_dir = "/content/drive/MyDrive/GRAVE_D_checkpoints/"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    dirpath=checkpoint_dir,
    filename="tft-grave-d-{epoch:02d}-{val_loss:.4f}",
    save_top_k=3,
    save_last=True,
    every_n_epochs=1,
    mode="min",
    verbose=True,
)
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,
    patience=8,
    verbose=True,
    mode="min",
)
csv_logger = CSVLogger(
    save_dir="/content/drive/MyDrive/GRAVE_D_logs/",
    name="tft_training_logs",
    version=None,
)

trainer = Trainer(
    max_epochs=50,
    accelerator="gpu",
    devices=1,
    precision="bf16-mixed",
    gradient_clip_val=1.0,
    accumulate_grad_batches=4,   # effective batch = 512 × 4 = 2048
    callbacks=[early_stop_callback, checkpoint_callback, RichProgressBar(leave=True)],
    logger=csv_logger,
    enable_model_summary=True,
    log_every_n_steps=25,
)

# --- LR Finder only if no checkpoint exists ---
last_ckpt = os.path.join(checkpoint_dir, "tft-grave-d-last.ckpt")

if os.path.exists(last_ckpt):
    print(f"Checkpoint found at {last_ckpt} — skipping LR finder.")
    print(f"Current LR: {tft.hparams.learning_rate}")
else:
    print("No checkpoint found — running LR finder once...")
    tuner = Tuner(trainer)
    try:
        lr_result = tuner.lr_find(tft, train_dataloaders=train_dataloader)
        suggested_lr = lr_result.suggestion()
        print(f"Suggested LR: {suggested_lr}")
        tft.hparams.learning_rate = suggested_lr

        import matplotlib.pyplot as plt
        fig = lr_result.plot(suggest=True)
        fig.savefig("/content/drive/MyDrive/GRAVE_D_logs/lr_finder_plot.png",
                    dpi=150, bbox_inches="tight")
        plt.close(fig)
        print("LR finder plot saved.")
    except Exception as e:
        print(f"LR finder failed: {e} — using default 0.001")
        tft.hparams.learning_rate = 0.001
    finally:
        trainer.callbacks = [cb for cb in trainer.callbacks if not isinstance(cb, LearningRateFinder)]
        torch.cuda.empty_cache()
        gc.collect()
        print(f"Post-LR-finder VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print(f"\nFinal LR: {tft.hparams.learning_rate}")
print(f"Batch size: {BATCH_SIZE} × {trainer.accumulate_grad_batches} = {BATCH_SIZE * trainer.accumulate_grad_batches} effective")
print("Trainer ready — proceed to Cell 11.")


INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.

No checkpoint found — running LR finder once...


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: `weights_only` was not set, defaulting to `False`.
INFO:lightning.pytorch.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

INFO: LR finder stopped early after 98 steps due to diverging loss.
INFO:lightning.pytorch.tuner.lr_finder:LR finder stopped early after 98 steps due to diverging loss.
INFO: Restoring states from the checkpoint path at /content/.lr_find_e9a3a446-ee2a-4c93-96c0-cb6529f788dc.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/.lr_find_e9a3a446-ee2a-4c93-96c0-cb6529f788dc.ckpt


LR finder failed: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more ab

In [15]:
# ---- Cell 11: Quick Stability Test ----
import torch, gc
from lightning.pytorch import Trainer

torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()
print(f"Pre-test VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

trainer_test = Trainer(
    fast_dev_run=2,
    accelerator="gpu",
    devices=1,
    precision="bf16-mixed",
    gradient_clip_val=1.0,
    accumulate_grad_batches=trainer.accumulate_grad_batches,
    logger=False,
    enable_progress_bar=True,
)

test_train_dl = train_dataset.to_dataloader(
    train=True, batch_size=64, num_workers=2
)
test_val_dl = val_dataset.to_dataloader(
    train=False, batch_size=64, num_workers=2
)

trainer_test.fit(tft, train_dataloaders=test_train_dl, val_dataloaders=test_val_dl)
print("\nStability test passed — safe to proceed to full training.")

del trainer_test, test_train_dl, test_val_dl
torch.cuda.empty_cache()
gc.collect()
print(f"Post-test VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: Running in `fast_dev_run` mode: will run the requested loop using 2 batch(es). Loggi

Pre-test VRAM: 0.01 GB


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MultiLoss                       │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    832 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  154 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  354 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  235 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  4.2 M │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  4.2 M │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  525 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │  1.0 K │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.3 M │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  656 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  526 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  526 K │ train │     0 │
│ 20 │ output_layer                       │ ModuleList                      │  5.6 K │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 18.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 18.0 M                                                                                               
Total estimated model params size (MB): 71                                                                         
Modules in train mode: 443                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.



Stability test passed — safe to proceed to full training.
Post-test VRAM: 0.01 GB


In [16]:
# ---- Cell 12: Full Training (with resume, no LR finder) ----
import torch, gc, os
import pandas as pd
from lightning.pytorch.callbacks import LearningRateFinder

torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

# Ensure LR finder callback is gone
trainer.callbacks = [cb for cb in trainer.callbacks
                     if not isinstance(cb, LearningRateFinder)]
print(f"Active callbacks: {[type(cb).__name__ for cb in trainer.callbacks]}")

torch.serialization.add_safe_globals([
    pd.core.internals.managers.BlockManager,
    pd._libs.internals._unpickle_block,
])

print(f"Pre-training VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Available VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
print(f"LR: {tft.hparams.learning_rate}")
print(f"Batch size: {train_dataloader.batch_size} × {trainer.accumulate_grad_batches} = {train_dataloader.batch_size * trainer.accumulate_grad_batches} effective")
print(f"Checkpoints → {checkpoint_callback.dirpath}")

last_ckpt = os.path.join(checkpoint_callback.dirpath, "tft-grave-d-last.ckpt")
if os.path.exists(last_ckpt):
    print(f"\nResuming from checkpoint: {last_ckpt}")
    ckpt_path = last_ckpt
else:
    print("\nStarting fresh training...")
    ckpt_path = None

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    ckpt_path=ckpt_path,
)

print(f"\nTraining complete!")
print(f"Best checkpoint: {checkpoint_callback.best_model_path}")
print(f"Last checkpoint: {checkpoint_callback.last_model_path}")


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Active callbacks: ['EarlyStopping', 'RichProgressBar', 'RichModelSummary', 'ModelCheckpoint']
Pre-training VRAM: 0.01 GB
Available VRAM: 101.97 GB
LR: 0.001
Batch size: 512 × 4 = 2048 effective
Checkpoints → /content/drive/MyDrive/GRAVE_D_checkpoints

Starting fresh training...


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MultiLoss                       │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    832 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  154 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  354 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  235 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  4.2 M │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  4.2 M │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  525 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │  1.0 K │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.3 M │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  656 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  526 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 M │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  526 K │ train │     0 │
│ 20 │ output_layer                       │ ModuleList                      │  5.6 K │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 18.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 18.0 M                                                                                               
Total estimated model params size (MB): 71                                                                         
Modules in train mode: 443                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()

INFO: Epoch 7, global step 8232: 'val_loss' was not in top 3
INFO:lightning.pytorch.utilities.rank_zero:Epoch 7, global step 8232: 'val_loss' was not in top 3


Output()

INFO: Monitored metric val_loss did not improve in the last 8 records. Best score: 1.067. Signaling Trainer to stop.
INFO:lightning.pytorch.callbacks.early_stopping:Monitored metric val_loss did not improve in the last 8 records. Best score: 1.067. Signaling Trainer to stop.
INFO: Epoch 8, global step 9261: 'val_loss' was not in top 3
INFO:lightning.pytorch.utilities.rank_zero:Epoch 8, global step 9261: 'val_loss' was not in top 3



Training complete!
Best checkpoint: /content/drive/MyDrive/GRAVE_D_checkpoints/tft-grave-d-epoch=00-val_loss=1.0671.ckpt
Last checkpoint: /content/drive/MyDrive/GRAVE_D_checkpoints/last.ckpt


In [29]:
# ---- Cell 13: Get stacked predictions ----
import os, gc, torch
import numpy as np
import pandas as pd

torch.cuda.empty_cache()
gc.collect()

CHECKPOINT_DIR = "/content/drive/MyDrive/GRAVE_D_checkpoints"
BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "tft-grave-d-epoch=00-val_loss=1.0671.ckpt",  # adjust if needed
)
print(f"Loading best checkpoint:\n{BEST_CHECKPOINT}")
assert os.path.exists(BEST_CHECKPOINT), "Best checkpoint path does not exist."

best_tft = tft.load_from_checkpoint(
    BEST_CHECKPOINT,
    map_location="cuda" if torch.cuda.is_available() else "cpu",
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_tft.to(device)
best_tft.eval()
print(f"Loaded model on {device}")

# Standard predict → list of arrays [batch_i, max_pred_len, n_targets]
preds_list = best_tft.predict(
    val_dataset,
    batch_size=512,
    num_workers=4,
    trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu"},
)
preds = np.concatenate(preds_list, axis=0)
print("preds shape:", preds.shape)  # (N, max_pred_len=8, 3)

# Save preds to reuse
PRED_SAVE_PATH = "/content/preds_val_dataset.npy"
np.save(PRED_SAVE_PATH, preds)
print(f"Saved predictions to {PRED_SAVE_PATH}")


Loading best checkpoint:
/content/drive/MyDrive/GRAVE_D_checkpoints/tft-grave-d-epoch=00-val_loss=1.0671.ckpt


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Loaded model on cuda
preds shape: (110016, 8)
Saved predictions to /content/preds_val_dataset.npy


In [31]:
# diagnostic probe #

raw = best_tft.predict(val_dataset, mode="raw", return_x=True, batch_size=4)
print(type(raw))
print(len(raw))
print(type(raw))
print(raw.keys())


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

<class 'pytorch_forecasting.models.base._base_model.Prediction'>
5
<class 'pytorch_forecasting.models.base._base_model.Prediction'>
('output', 'x', 'index', 'decoder_lengths', 'y')
